In [1]:
!pip install pyreaddbc dbfread pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 569.8 kB/s eta 0:00:00


In [ ]:
drive.mount('/content/drive')

In [2]:
import datetime
import hashlib
import logging
import os
from pathlib import Path
import re
import shutil
import struct
import tempfile
import time
import urllib.request
import numpy as np
import pandas as pd

# 1. CONECTAR GOOGLE DRIVE (só quando estiver rodando no Colab)
try:
  from google.colab import drive

  drive.mount('/content/drive')
except ImportError:
  pass

# 2. CONFIGURAÇÃO DE LOGS LIMPOS E ESTRUTURADOS
logging.basicConfig(
    level=logging.INFO, format='%(levelname)s [%(name)s]: %(message)s'
)
log = logging.getLogger('sinan_pipeline')

# 3. CAMINHO DOS DIRETÓRIOS NO GOOGLE DRIVE
DIRETORIO_BASE = Path(
    os.environ.get(
        'SINAN_DIRETORIO_BASE', '/content/drive/MyDrive/projeto_escorpioes/sinan'
    )
)
PASTA_BRONZE_DBC = DIRETORIO_BASE / '1_bronze'
PASTA_SILVER = DIRETORIO_BASE / '2_silver'
PASTA_TABELAS = DIRETORIO_BASE / 'tabelas'  # municipios_ibge.csv e cbo.csv

PASTA_SILVER.mkdir(parents=True, exist_ok=True)
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)

# Tabelas completas de apoio (baixadas automaticamente se não estiverem na pasta)
URL_MUNICIPIOS = (
    'https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/'
    'csv/municipios.csv'
)
URL_CBO = (
    'https://raw.githubusercontent.com/lucassmacedo/cbo-brasil/master/csv/'
    'CBO2002%20-%20Ocupacao.csv'
)

# --------------------------------------------------------------------------
# CONFIGURAÇÕES E DICIONÁRIOS EXPANDIDOS (2016-2026 E REGIÃO)
# --------------------------------------------------------------------------
ANOS_VALIDOS = range(2016, 2027)

COLUNAS_REMOVER = [
    'TP_NOT',
    'ID_AGRAVO',
    'SG_UF_NOT',
    'ID_REGIONA',
    'SG_UF',
    'ID_RG_RESI',
    'ID_PAIS',
    'ANT_UF',
    'CLI_OUTR_2',
    'CLI_OUTR_3',
    'ANI_TIPO_1',
    'ANI_SERPEN',
    'ANI_ARANHA',
    'ANI_LAGART',
    'NU_AMPOLAS',
    'NU_AMPOL_1',
    'NU_AMPOL_8',
    'NU_AMPOL_6',
    'NU_AMPOL_4',
    'NU_AMPO_7',
    'NU_AMPO_5',
    'NU_AMPOL_3',
    'DT_DIGITA',
]

COLUNAS_SEMANA = ['SEM_NOT', 'SEM_PRI']
COLUNAS_MUNICIPIO = ['ID_MUNICIP', 'ID_MN_RESI', 'ANT_MUNIC_']
COLUNA_OCUPACAO = 'ID_OCUPA_N'
COLUNAS_DATA = [
    'DT_NOTIFIC',
    'DT_SIN_PRI',
    'DT_INVEST',
    'ANT_DT_ACI',
    'DT_OBITO',
    'DT_ENCERRA',
]

# DICIONÁRIO DE MUNICÍPIOS EXPANDIDO PARA COTIA E REGIÃO VIZINHA
MUNICIPIOS_IBGE = {
    '350570': 'Barueri',
    '351060': 'Carapicuíba',
    '351300': 'Cotia',
    '351500': 'Embu das Artes',
    '352220': 'Itapecerica da Serra',
    '352250': 'Itapevi',
    '352500': 'Jandira',
    '353440': 'Osasco',
    '355030': 'São Paulo',
    '355060': 'São Roque',
    '355280': 'Taboão da Serra',
    '355645': 'Vargem Grande Paulista',
}

# Códigos especiais do SINAN (não existem na CBO) - têm prioridade sobre a CBO.
# A tabela CBO completa (2.614 ocupações) é carregada de tabelas/cbo.csv.
OCUPACOES_CBO = {
    '000000': 'Ignorado',
    '999991': 'Estudante',
    '999992': 'Dona de casa',
    '999993': 'Aposentado',
    '999994': 'Desempregado',
    '999995': 'Detento',
    '998999': 'Ignorado',
}

SIM_NAO = {'1': 'Sim', '2': 'Não', '9': 'Ignorado'}

COLUNAS_SIM_NAO = [
    'MCLI_LOCAL',
    'CLI_DOR',
    'CLI_EDEMA',
    'CLI_EQUIMO',
    'CLI_NECROS',
    'CLI_LOCAL_',
    'MCLI_SIST',
    'CLI_NEURO',
    'CLI_HEMORR',
    'CLI_VAGAIS',
    'CLI_MIOLIT',
    'CLI_RENAL',
    'CON_SOROTE',
    'COM_LOC',
    'COM_SECUND',
    'COM_NECROS',
    'COM_COMPOR',
    'COM_DEFICT',
    'COM_APUTAC',
    'COM_SISTEM',
    'COM_RENAL',
    'COM_EDEMA',
    'COM_SEPTIC',
    'COM_CHOQUE',
    'DOENCA_TRA',
]

CATEGORIAS = {
    'CS_SEXO': {'M': 'Masculino', 'F': 'Feminino', 'I': 'Ignorado'},
    'CS_GESTANT': {
        '1': '1º Trimestre',
        '2': '2º Trimestre',
        '3': '3º Trimestre',
        '4': 'Idade gestacional ignorada',
        '5': 'Não',
        '6': 'Não se aplica',
        '9': 'Ignorado',
    },
    'CS_RACA': {
        '1': 'Branca',
        '2': 'Preta',
        '3': 'Amarela',
        '4': 'Parda',
        '5': 'Indígena',
        '9': 'Ignorado',
    },
    'CS_ESCOL_N': {
        '0': 'Analfabeto',
        '1': '1ª a 4ª série incompleta do EF',
        '2': '4ª série completa do EF',
        '3': '5ª a 8ª série incompleta do EF',
        '4': 'Ensino fundamental completo',
        '5': 'Ensino médio incompleto',
        '6': 'Ensino médio completo',
        '7': 'Educação superior incompleta',
        '8': 'Educação superior completa',
        '9': 'Ignorado',
        '10': 'Não se aplica',
    },
    'ANT_TEMPO_': {
        '1': '0 a 1h',
        '2': '1 a 3h',
        '3': '3 a 6h',
        '4': '6 a 12h',
        '5': '12 a 24h',
        '6': '24h ou mais',
        '9': 'Ignorado',
    },
    'ANT_LOCA_1': {
        '1': 'Cabeça',
        '2': 'Braço',
        '3': 'Antebraço',
        '4': 'Mão',
        '5': 'Dedo da mão',
        '6': 'Tronco',
        '7': 'Coxa',
        '8': 'Perna',
        '9': 'Pé',
        '10': 'Dedo do pé',
        '99': 'Ignorado',
    },
    'CLI_TEMPO_': {
        '1': 'Normal',
        '2': 'Alterado',
        '9': 'Não realizado',
    },
    'TP_ACIDENT': {
        '1': 'Serpente',
        '2': 'Aranha',
        '3': 'Escorpião',
        '4': 'Lagarta',
        '5': 'Abelha',
        '6': 'Outros',
        '9': 'Ignorado',
    },
    'TRA_CLASSI': {
        '1': 'Leve',
        '2': 'Moderado',
        '3': 'Grave',
        '9': 'Ignorado',
    },
    'EVOLUCAO': {
        '1': 'Cura',
        '2': 'Óbito por acidente por animal peçonhento',
        '3': 'Óbito por outra causa',
        '9': 'Ignorado',
    },
}
CATEGORIAS.update({c: SIM_NAO for c in COLUNAS_SIM_NAO})

COLUNAS_NUMERICAS = ['NU_ANO', 'ANO_NASC', 'NU_AMPOL_9']

NOMES_COLUNAS = {
    'DT_NOTIFIC': 'Data da Notificação',
    'SEM_NOT': 'Semana da Notificação',
    'NU_ANO': 'Ano da Notificação',
    'ID_MUNICIP': 'Município de Notificação',
    'DT_SIN_PRI': 'Data dos Primeiros Sintomas',
    'SEM_PRI': 'Semana dos Primeiros Sintomas',
    'ANO_NASC': 'Ano de Nascimento',
    'NU_IDADE_N': 'Idade',
    'CS_SEXO': 'Sexo',
    'CS_GESTANT': 'Gestante',
    'CS_RACA': 'Raça/Cor',
    'CS_ESCOL_N': 'Escolaridade',
    'ID_MN_RESI': 'Município de Residência',
    'DT_INVEST': 'Data da Investigação',
    'ID_OCUPA_N': 'Ocupação',
    'ANT_DT_ACI': 'Data do Acidente',
    'ANT_MUNIC_': 'Município de Ocorrência',
    'ANT_TEMPO_': 'Tempo entre Picada e Atendimento',
    'ANT_LOCA_1': 'Local da Picada',
    'MCLI_LOCAL': 'Manifestação Local',
    'CLI_DOR': 'Dor',
    'CLI_EDEMA': 'Edema',
    'CLI_EQUIMO': 'Equimose',
    'CLI_NECROS': 'Necrose',
    'CLI_LOCAL_': 'Outras Manifestações Locais',
    'CLI_LOCA_1': 'Especificação de Outras Manifestações Locais',
    'MCLI_SIST': 'Manifestação Sistêmica',
    'CLI_NEURO': 'Neuroparalítica',
    'CLI_HEMORR': 'Hemorrágica',
    'CLI_VAGAIS': 'Vagal',
    'CLI_MIOLIT': 'Miolítica/Hemolítica',
    'CLI_RENAL': 'Renal',
    'CLI_TEMPO_': 'Tempo de Coagulação',
    'TP_ACIDENT': 'Tipo de Animal',
    'TRA_CLASSI': 'Classificação do Caso',
    'CON_SOROTE': 'Soroterapia Aplicada',
    'NU_AMPOL_9': 'Nº de Ampolas de Soro Antiescorpiônico',
    'COM_LOC': 'Complicação Local',
    'COM_SECUND': 'Infecção Secundária',
    'COM_NECROS': 'Necrose Extensa',
    'COM_COMPOR': 'Síndrome Comportamental',
    'COM_DEFICT': 'Déficit Funcional',
    'COM_APUTAC': 'Amputação',
    'COM_SISTEM': 'Complicação Sistêmica',
    'COM_RENAL': 'Insuficiência Renal',
    'COM_EDEMA': 'Insuficiência Respiratória/Edema Pulmonar',
    'COM_SEPTIC': 'Septicemia',
    'COM_CHOQUE': 'Choque',
    'DOENCA_TRA': 'Acidente Relacionado ao Trabalho',
    'EVOLUCAO': 'Evolução do Caso',
    'DT_OBITO': 'Data do Óbito',
    'DT_ENCERRA': 'Data de Encerramento',
}


# --------------------------------------------------------------------------
# FUNÇÕES DE TRATAMENTO
# --------------------------------------------------------------------------
def _normalizar_codigo(serie: pd.Series, tamanho: int = 6) -> pd.Series:
  s = serie.astype('string').str.strip()
  s = s.str.replace(r'\.0+$', '', regex=True)
  s = s.str.replace(r'\D', '', regex=True)
  s = s.where(s.str.len() > 0)
  s = s.str.slice(0, tamanho)
  return s.str.zfill(tamanho)


UFS_IBGE = {
    '11': 'RO', '12': 'AC', '13': 'AM', '14': 'RR', '15': 'PA', '16': 'AP',
    '17': 'TO', '21': 'MA', '22': 'PI', '23': 'CE', '24': 'RN', '25': 'PB',
    '26': 'PE', '27': 'AL', '28': 'SE', '29': 'BA', '31': 'MG', '32': 'ES',
    '33': 'RJ', '35': 'SP', '41': 'PR', '42': 'SC', '43': 'RS', '50': 'MS',
    '51': 'MT', '52': 'GO', '53': 'DF',
}


def _ler_tabela(nome_arquivo: str, url: str, col_codigo: str, col_nome: str):
  """Lê a tabela de apoio da pasta 'tabelas' do Drive; se não existir, baixa
  do GitHub e salva lá para as próximas execuções."""
  caminho = PASTA_TABELAS / nome_arquivo
  if not caminho.exists():
    log.info(f'Baixando tabela de apoio {nome_arquivo}...')
    urllib.request.urlretrieve(url, caminho)
  tab = pd.read_csv(caminho, sep=None, engine='python', dtype=str,
                    encoding='utf-8-sig')
  tab.columns = [c.strip().lower() for c in tab.columns]
  # aceita tanto o formato original do GitHub quanto o salvo (codigo;nome)
  col_codigo = col_codigo if col_codigo in tab.columns else 'codigo'
  col_nome = col_nome if col_nome in tab.columns else 'nome'
  tab[col_nome] = tab[col_nome].str.strip().str.replace(r'\s+', ' ', regex=True)
  return dict(zip(tab[col_codigo].str.strip(), tab[col_nome]))


def carregar_tabela_municipios() -> dict[str, str]:
  """Todos os 5.570 municípios (código IBGE de 6 dígitos -> nome).
  Municípios de SP aparecem só com o nome; os de outros estados levam a UF
  ('Salvador/BA') para evitar nomes repetidos entre estados."""
  bruta = _ler_tabela('municipios_ibge.csv', URL_MUNICIPIOS, 'codigo_ibge',
                      'nome')
  tabela = {}
  for cod7, nome in bruta.items():
    cod6 = cod7[:6]
    uf = UFS_IBGE.get(cod6[:2], '')
    tabela[cod6] = nome if uf in ('SP', '') else f'{nome}/{uf}'
  # Códigos terminados em 0000 = "município ignorado" dentro da UF
  for cod_uf, sigla in UFS_IBGE.items():
    tabela[f'{cod_uf}0000'] = f'Município ignorado ({sigla})'
  tabela.update(MUNICIPIOS_IBGE)  # nomes da região têm prioridade
  return tabela


def carregar_tabela_cbo() -> dict[str, str]:
  """CBO 2002 completa (2.614 ocupações) + códigos especiais do SINAN."""
  tabela = _ler_tabela('cbo.csv', URL_CBO, 'code', 'name')
  tabela.update(OCUPACOES_CBO)  # códigos especiais (9999xx...) têm prioridade
  return tabela


TABELA_MUNICIPIOS = carregar_tabela_municipios()
TABELA_CBO = carregar_tabela_cbo()


def _resumir(codigos, limite=15):
  """Lista curta para o log (evita avisos gigantes)."""
  extra = f' ... (+{len(codigos) - limite})' if len(codigos) > limite else ''
  return ', '.join(codigos[:limite]) + extra


def _mapear_ocupacao(serie: pd.Series, tabela: dict[str, str]) -> pd.Series:
  codigos = _normalizar_codigo(serie, tamanho=6)
  nomes = codigos.map(tabela)

  faltantes = sorted(set(codigos[nomes.isna() & codigos.notna()].dropna()))
  if faltantes:
    log.warning(
        '⚠️ LOG DE OCUPAÇÕES: códigos fora da CBO 2002 (rotulados como'
        f' "não catalogada"): {_resumir(faltantes)}'
    )

  nomes = nomes.fillna(
      codigos.apply(
          lambda c: f'Ocupação não catalogada (CBO {c})'
          if pd.notna(c)
          else 'Não informado'
      )
  )
  return nomes.fillna('Não informado')


def _mapear(serie: pd.Series, tabela: dict[str, str], rotulo: str) -> pd.Series:
  codigos = _normalizar_codigo(serie)
  nomes = codigos.map(tabela)
  faltantes = sorted(set(codigos[nomes.isna() & codigos.notna()].dropna()))
  if faltantes:
    log.warning(
        f'⚠️ LOG DE MUNICÍPIOS ({rotulo}): códigos fora da tabela IBGE'
        f' (rotulados como "não catalogado"): {_resumir(faltantes)}'
    )
  nomes = nomes.fillna(
      codigos.apply(
          lambda c: f'Município não catalogado ({c})'
          if pd.notna(c)
          else 'Não informado'
      )
  )
  return nomes.fillna('Não informado')


def _decodificar(
    serie: pd.Series, tabela: dict[str, str], coluna: str
) -> pd.Series:
  chave = serie.astype('string').str.strip().str.upper()
  chave = chave.str.replace(r'\.0+$', '', regex=True)
  chave = chave.str.replace(r'^0+(?=\d)', '', regex=True)
  nomes = chave.map(tabela)

  faltantes = sorted(set(chave[nomes.isna() & chave.notna()].dropna()))
  if faltantes:
    log.info(
        f'⚠️ LOG DE CATEGORIAS ({coluna}): Valores fora do dicionário'
        f' mantidos: {", ".join(faltantes)}'
    )

  res = nomes.fillna(serie.astype('string').str.strip())
  return res.replace('', 'Não informado').fillna('Não informado')


_UNIDADES_IDADE = {
    '1': ('hora', 'horas'),
    '2': ('dia', 'dias'),
    '3': ('mês', 'meses'),
    '4': ('ano', 'anos'),
}


def _formatar_idade(serie: pd.Series) -> pd.Series:
  def conv(v):
    if pd.isna(v) or str(v).strip() in ('', 'nan', 'None'):
      return 'Não informado'
    m = re.fullmatch(r'([1-4])(\d{3})', str(v).strip())
    if not m:
      return str(v).strip()
    qtd = int(m.group(2))
    sing, plur = _UNIDADES_IDADE[m.group(1)]
    return f'{qtd} {sing if qtd == 1 else plur}'

  return serie.map(conv).astype('string').fillna('Não informado')


def _formatar_semana(serie: pd.Series) -> pd.Series:
  digitos = serie.astype('string').str.replace(r'\D', '', regex=True)
  semana = digitos.where(digitos.str.len() < 5, digitos.str.slice(-2))
  semana = pd.to_numeric(semana, errors='coerce').astype('Int64')
  invalidas = semana.notna() & ~semana.between(1, 53)
  if invalidas.any():
    semana = semana.mask(invalidas)
  return semana


_FORMATOS_DATA = ('%Y-%m-%d', '%d/%m/%Y', '%Y%m%d', '%d-%m-%Y', '%Y/%m/%d')


def _formatar_data(serie: pd.Series) -> pd.Series:
  s = serie.astype('string').str.strip().str.slice(0, 10)
  resultado = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
  for fmt in _FORMATOS_DATA:
    tentativa = pd.to_datetime(s, format=fmt, errors='coerce')
    resultado = resultado.fillna(tentativa.astype('datetime64[ns]'))
  return resultado.dt.strftime('%Y-%m-%d').fillna('Não informado')


def tratar_dataframe(df: pd.DataFrame) -> pd.DataFrame:
  df = df.copy()
  df.columns = [c.strip().upper() for c in df.columns]

  df = df.drop(columns=[c for c in COLUNAS_REMOVER if c in df.columns])

  for col in COLUNAS_SEMANA:
    if col in df.columns:
      df[col] = _formatar_semana(df[col])

  for col in COLUNAS_MUNICIPIO:
    if col in df.columns:
      df[col] = _mapear(df[col], TABELA_MUNICIPIOS, col)

  if COLUNA_OCUPACAO in df.columns:
    df[COLUNA_OCUPACAO] = _mapear_ocupacao(df[COLUNA_OCUPACAO], TABELA_CBO)

  for col in COLUNAS_DATA:
    if col in df.columns:
      df[col] = _formatar_data(df[col])

  for col, tabela in CATEGORIAS.items():
    if col in df.columns:
      df[col] = _decodificar(df[col], tabela, col)

  if 'NU_IDADE_N' in df.columns:
    df['NU_IDADE_N'] = _formatar_idade(df['NU_IDADE_N'])

  for col in COLUNAS_NUMERICAS:
    if col in df.columns:
      df[col] = (
          pd.to_numeric(df[col], errors='coerce')
          .fillna(0)
          .astype('Int64')
          .astype(str)
      )

  if 'CLI_LOCA_1' in df.columns:
    df['CLI_LOCA_1'] = (
        df['CLI_LOCA_1']
        .astype('string')
        .str.strip()
        .str.capitalize()
        .fillna('Não informado')
    )

  df = df.fillna('Não informado')
  df = df.rename(columns=NOMES_COLUNAS)
  return df


# --------------------------------------------------------------------------
# EXECUÇÃO DA PIPELINE UNIFICADA (COTIA E MUNICÍPIOS VIZINHOS)
# --------------------------------------------------------------------------
# --------------------------------------------------------------------------
# LEITURA RÁPIDA DO DBF (filtra os BYTES antes de decodificar)
# --------------------------------------------------------------------------
# O DBF é uma tabela de registros de tamanho fixo. Em vez de converter os
# ~250 mil registros do Brasil inteiro em dicionários Python (o que o dbfread
# faz, linha por linha, e é o que demorava), lemos o arquivo como uma matriz
# de bytes, filtramos escorpião + municípios da região direto nos bytes
# (numpy) e só decodificamos as poucas linhas que sobraram.
TIPO_ACIDENTE_ESCORPIAO = '3'
TIPOS_DBF_SUPORTADOS = set('CDNFL')


def _ler_cabecalho_dbf(f):
  cab = f.read(32)
  n_reg, tam_cab, tam_reg = struct.unpack('<IHH', cab[4:12])
  campos, off = [], 1  # byte 0 de cada registro = marca de exclusão
  while True:
    d = f.read(32)
    if not d or d[0] == 0x0D:
      break
    nome = d[:11].split(b'\x00')[0].decode('ascii', 'ignore').upper()
    campos.append((nome, chr(d[11]), d[16], off))
    off += d[16]
  return n_reg, tam_cab, tam_reg, campos


def _coluna_bytes(dados, off, tam):
  return np.ascontiguousarray(dados[:, off : off + tam]).view(f'S{tam}').ravel()


def _valor_dbf(tipo, texto):
  """Mesmas regras de conversão do dbfread (C, D, N, F, L)."""
  if tipo == 'C':
    return texto.rstrip('\x00 ')
  if tipo == 'D':
    if texto.strip(' 0') == '':
      return None
    try:
      return datetime.date(
          int(texto[:4]), int(texto[4:6]), int(texto[6:8])
      ).isoformat()
    except ValueError:
      return None
  if tipo in 'NF':
    t = texto.strip().strip('*')
    if not t.strip():
      return None
    try:
      return int(t)
    except ValueError:
      return float(t.replace(',', '.'))
  if tipo == 'L':
    return True if texto in 'TtYy' else False if texto in 'FfNn' else None
  return texto


def ler_dbf_filtrado(caminho, codigos, tp_acidente=TIPO_ACIDENTE_ESCORPIAO,
                     encoding='iso-8859-1'):
  """Lê o .dbf e devolve só os registros de `tp_acidente` cujo município de
  notificação OU de residência esteja em `codigos`."""
  with open(caminho, 'rb') as f:
    n_reg, tam_cab, tam_reg, campos = _ler_cabecalho_dbf(f)
    f.seek(tam_cab)
    dados = np.fromfile(f, dtype=np.uint8, count=n_reg * tam_reg)

  invalidos = [c for c in campos if c[1] not in TIPOS_DBF_SUPORTADOS]
  if invalidos:
    raise NotImplementedError(f'Tipos de campo não suportados: {invalidos[:3]}')
  n = len(dados) // tam_reg
  dados = dados[: n * tam_reg].reshape(n, tam_reg)
  idx = {nome: (off, tam) for nome, _, tam, off in campos}
  for obrig in ('TP_ACIDENT', 'ID_MUNICIP', 'ID_MN_RESI'):
    if obrig not in idx:
      raise KeyError(f'Campo {obrig} ausente no DBF')

  # 1º filtro (barato, sobre tudo): registro não excluído + tipo de acidente
  tp = np.char.strip(_coluna_bytes(dados, *idx['TP_ACIDENT']))
  dados = dados[(dados[:, 0] == 0x20) & (tp == tp_acidente.encode())]

  # 2º filtro (só nos que sobraram): município de notificação OU residência
  alvo = np.array(sorted(codigos), dtype='S6')
  def _cod6(nome):
    return np.char.zfill(np.char.strip(_coluna_bytes(dados, *idx[nome])), 6)
  dados = dados[
      np.isin(_cod6('ID_MUNICIP'), alvo) | np.isin(_cod6('ID_MN_RESI'), alvo)
  ]

  # Decodifica apenas as linhas restantes
  colunas = {}
  for nome, tipo, tam, off in campos:
    brutos = _coluna_bytes(dados, off, tam).tolist()
    colunas[nome] = [_valor_dbf(tipo, b.decode(encoding)) for b in brutos]
  df = pd.DataFrame(colunas)
  df['ID_MUNICIP'] = df['ID_MUNICIP'].astype(str).str.strip().str.zfill(6)
  df['ID_MN_RESI'] = df['ID_MN_RESI'].astype(str).str.strip().str.zfill(6)
  df['TP_ACIDENT'] = df['TP_ACIDENT'].astype(str).str.strip()
  return df


def _ler_filtrado_dbfread(caminho, codigos, tp_acidente=TIPO_ACIDENTE_ESCORPIAO):
  """Plano B (lento): método antigo, usado só se o leitor rápido falhar."""
  from dbfread import DBF

  df = pd.DataFrame(iter(DBF(str(caminho), encoding='iso-8859-1')))
  df['ID_MUNICIP'] = df['ID_MUNICIP'].astype(str).str.strip().str.zfill(6)
  df['ID_MN_RESI'] = df['ID_MN_RESI'].astype(str).str.strip().str.zfill(6)
  df['TP_ACIDENT'] = df['TP_ACIDENT'].astype(str).str.strip()
  return df[
      (df['ID_MUNICIP'].isin(codigos) | df['ID_MN_RESI'].isin(codigos))
      & (df['TP_ACIDENT'] == tp_acidente)
  ].copy()


# --------------------------------------------------------------------------
# CACHE POR ARQUIVO (re-execuções só processam .dbc novos ou alterados)
# --------------------------------------------------------------------------
PASTA_CACHE = PASTA_SILVER / '_cache'


def _caminho_cache(arq_dbc: Path, codigos) -> Path:
  st = arq_dbc.stat()
  chave = f'{arq_dbc.name}|{st.st_size}|{st.st_mtime_ns}|{sorted(codigos)}'
  chave += f'|{TIPO_ACIDENTE_ESCORPIAO}|v1'
  return PASTA_CACHE / f'{arq_dbc.stem}_{hashlib.md5(chave.encode()).hexdigest()[:10]}.parquet'


def _extrair_filtrado(arq_dbc: Path, codigos, usar_cache=True) -> pd.DataFrame:
  """.dbc -> registros de escorpião da região (bruto, antes do tratamento)."""
  import pyreaddbc

  cache = _caminho_cache(arq_dbc, codigos)
  if usar_cache and cache.exists():
    try:
      df = pd.read_parquet(cache)
      print(f'   ⚡ cache: {len(df)} notificações (pulou descompressão/leitura)')
      return df
    except Exception as e:  # cache corrompido -> refaz
      log.warning(f'Cache ilegível ({cache.name}): {e}')

  # .dbf temporário no disco LOCAL do Colab (rápido), não no Drive (lento)
  pasta_tmp = Path(tempfile.mkdtemp(prefix='sinan_'))
  try:
    dbf_tmp = pasta_tmp / f'{arq_dbc.stem}.dbf'
    t0 = time.time()
    pyreaddbc.dbc2dbf(str(arq_dbc), str(dbf_tmp))
    t1 = time.time()
    try:
      df = ler_dbf_filtrado(dbf_tmp, codigos)
    except Exception as e:
      log.warning(f'Leitor rápido falhou ({e}); usando dbfread (lento).')
      df = _ler_filtrado_dbfread(dbf_tmp, codigos)
    t2 = time.time()
    print(f'   ⏱ descompressão {t1 - t0:.0f}s | leitura+filtro {t2 - t1:.0f}s')
  finally:
    shutil.rmtree(pasta_tmp, ignore_errors=True)

  if usar_cache:
    try:
      PASTA_CACHE.mkdir(parents=True, exist_ok=True)
      df.to_parquet(cache, index=False)
    except Exception as e:
      log.warning(f'Não foi possível gravar o cache: {e}')
  return df


def executar_pipeline_sinan_unificada(usar_cache=True):
  arquivos_dbc = sorted(
      list(PASTA_BRONZE_DBC.glob('*.dbc')) + list(PASTA_BRONZE_DBC.glob('*.DBC'))
  )

  if not arquivos_dbc:
    print(
        f"❌ Nenhum arquivo .dbc encontrado em '{PASTA_BRONZE_DBC}'. Verifique"
        ' as pastas do seu Drive.'
    )
    return

  print(
      f'🚀 Iniciando Pipeline SINAN Regional (2016-2026): {len(arquivos_dbc)}'
      ' arquivos .dbc encontrados.\n'
  )

  dfs_tratados = []
  CODIGOS_REGIONAL = set(MUNICIPIOS_IBGE.keys())
  t_inicio = time.time()

  for i, arq_dbc in enumerate(arquivos_dbc, 1):
    print(f'[{i}/{len(arquivos_dbc)}] ⏳ Processando: {arq_dbc.name}...')
    df_filtrado = _extrair_filtrado(arq_dbc, CODIGOS_REGIONAL, usar_cache)
    print(f'   -> Extraídas {len(df_filtrado)} notificações na região.')

    if not df_filtrado.empty:
      dfs_tratados.append(tratar_dataframe(df_filtrado))

  if dfs_tratados:
    base_silver = pd.concat(dfs_tratados, ignore_index=True).drop_duplicates()

    if 'Data da Notificação' in base_silver.columns:
      base_silver = base_silver.sort_values('Data da Notificação')

    caminho_final_silver = (
        PASTA_SILVER / 'sinan_cotia_regiao_escorpioes_2016_2026_silver.csv'
    )
    base_silver.to_csv(
        caminho_final_silver, index=False, encoding='utf-8-sig', sep=';'
    )

    print('\n' + '=' * 70)
    print('🎉 PIPELINE REGIONAL CONCLUÍDA COM SUCESSO!')
    print(f'📍 Base Silver Consolidada salva em: {caminho_final_silver}')
    print(
        f'📊 Total Final: {len(base_silver)} notificações de escorpião na'
        ' região (2016-2026).'
    )
    print(f'⏱ Tempo total: {(time.time() - t_inicio) / 60:.1f} min')
    print('=' * 70)
    return base_silver


def corrigir_silver_existente(caminho_entrada, caminho_saida=None):
  """Corrige uma base silver JÁ gerada (sem precisar refazer os .dbc):
  converte códigos IBGE que sobraram nas colunas de município e os
  'CBO 123456' que sobraram em Ocupação."""
  df = pd.read_csv(caminho_entrada, sep=';', dtype=str, encoding='utf-8-sig',
                   keep_default_na=False)
  for col in ['Município de Notificação', 'Município de Residência',
              'Município de Ocorrência']:
    codigo = df[col].str.fullmatch(r'\d{5,7}')
    df.loc[codigo, col] = _mapear(df.loc[codigo, col], TABELA_MUNICIPIOS, col)
  sobra = df['Ocupação'].str.fullmatch(r'CBO \d+')
  df.loc[sobra, 'Ocupação'] = _mapear_ocupacao(
      df.loc[sobra, 'Ocupação'].str.replace('CBO ', '', regex=False),
      TABELA_CBO,
  )
  caminho_saida = caminho_saida or str(caminho_entrada).replace(
      '.csv', '_corrigido.csv'
  )
  df.to_csv(caminho_saida, index=False, encoding='utf-8-sig', sep=';')
  return df


if __name__ == '__main__':
  executar_pipeline_sinan_unificada()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Iniciando Pipeline SINAN Regional (2016-2026): 11 arquivos .dbc encontrados.

[1/11] ⏳ Processando: ANIMBR16.dbc...
   ⏱ descompressão 2s | leitura+filtro 0s


   -> Extraídas 333 notificações na região.
[2/11] ⏳ Processando: ANIMBR17.dbc...
   ⏱ descompressão 3s | leitura+filtro 0s


   -> Extraídas 401 notificações na região.
[3/11] ⏳ Processando: ANIMBR18.dbc...
   ⏱ descompressão 2s | leitura+filtro 0s
   -> Extraídas 557 notificações na região.
[4/11] ⏳ Processando: ANIMBR19.dbc...


   ⏱ descompressão 2s | leitura+filtro 0s
   -> Extraídas 542 notificações na região.
[5/11] ⏳ Processando: ANIMBR20.dbc...


   ⏱ descompressão 2s | leitura+filtro 0s
   -> Extraídas 654 notificações na região.
[6/11] ⏳ Processando: ANIMBR21.dbc...
   ⏱ descompressão 2s | leitura+filtro 0s


   -> Extraídas 632 notificações na região.
[7/11] ⏳ Processando: ANIMBR22.dbc...
   ⏱ descompressão 3s | leitura+filtro 0s


   -> Extraídas 712 notificações na região.
[8/11] ⏳ Processando: ANIMBR23.dbc...
   ⏱ descompressão 3s | leitura+filtro 1s
   -> Extraídas 839 notificações na região.


[9/11] ⏳ Processando: ANIMBR24.dbc...


   ⏱ descompressão 2s | leitura+filtro 0s
   -> Extraídas 779 notificações na região.
[10/11] ⏳ Processando: ANIMBR25.dbc...
   ⏱ descompressão 2s | leitura+filtro 0s


   -> Extraídas 1081 notificações na região.
[11/11] ⏳ Processando: ANIMBR26.dbc...


   ⏱ descompressão 2s | leitura+filtro 0s
   -> Extraídas 653 notificações na região.

🎉 PIPELINE REGIONAL CONCLUÍDA COM SUCESSO!
📍 Base Silver Consolidada salva em: /content/drive/MyDrive/projeto_escorpioes/sinan/2_silver/sinan_cotia_regiao_escorpioes_2016_2026_silver.csv
📊 Total Final: 7181 notificações de escorpião na região (2016-2026).
⏱ Tempo total: 0.6 min
